# Project 2: CNN Image Classification — EuroSAT Satellite Imagery

## Land-Use / Land-Cover Classification with Convolutional Neural Networks

**Project Title:** Satellite Land-Use Classification with CNNs (EuroSAT)

**Rationale:** Accurate land-use and land-cover (LULC) mapping from satellite imagery is crucial for urban planning, environmental monitoring, and precision agriculture. The EuroSAT benchmark (Helber et al., 2019) provides 27 000 Sentinel-2 RGB patches across 10 land-use classes, making it a challenging and socially relevant CNN classification task that is completely distinct from CIFAR-10 and generic natural-object datasets.

**Why CNNs?** Satellite patches share the same spatial-pattern structure (textures, edges, repeated crop rows, building grids) that CNNs were designed to exploit via learned local filters. Transfer learning from ImageNet also transfers well because low-level edge/texture detectors are domain-agnostic.

---

### Rubric Coverage
| # | Rubric Item | Section |
|---|---|---|
| 1 | Dataset Preparation & Data Augmentation (rotation, flip, scale) | §2–§3 |
| 2 | CNN Model Design — custom + transfer learning | §4 |
| 3 | Model Training with configurable optimizer | §5 |
| 4 | Performance Evaluation (accuracy, precision, recall, F1, confusion matrix, learning curves) | §6 |

### Experimental Plan (for report)
Run the following controlled experiments and tabulate results:
1. **Baseline** — custom CNN, no augmentation, Adam, lr=1e-3
2. **Augmentation effect** — custom CNN + augmentation vs. baseline
3. **Architecture** — MobileNetV2 transfer learning vs. custom CNN (with augmentation)
4. **Optimizer** — Adam vs. SGD+momentum on best architecture
5. **Fine-tuning** — partially unfreeze MobileNetV2 backbone and compare

---

### Dataset Download
EuroSAT RGB is freely available via TensorFlow Datasets (no account required):  
```python
import tensorflow_datasets as tfds
ds = tfds.load('eurosat/rgb', split='train', as_supervised=True)
```
Or download the ZIP manually from:  
https://zenodo.org/record/7711810 (EuroSAT_RGB.zip, ~90 MB)

## 1) Import Libraries & Global Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import json
import time
import warnings
import itertools
from pathlib import Path

# ── Numerical / Plotting ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Deep-learning stack (TensorFlow / Keras) ──────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import (
    MobileNetV2, ResNet50V2, VGG16, EfficientNetB0
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger
)

# ── Metrics ───────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('tab10')

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# ── Global hyper-parameters (change here for experiments) ─────────────────────
IMG_SIZE    = (64, 64)          # EuroSAT native resolution
IMG_SHAPE   = IMG_SIZE + (3,)   # RGB
BATCH_SIZE  = 32
NUM_CLASSES = 10
EPOCHS      = 30                # reduce for quick demo; 50+ for full training

# Configurable optimizer — swap 'adam' / 'sgd' / 'rmsprop' for experiments
OPTIMIZER_NAME = 'adam'         # <-- change for optimizer experiment
LEARNING_RATE  = 1e-3

# Transfer-learning backbone — swap 'mobilenet' / 'resnet' / 'vgg' / 'efficientnet'
BACKBONE       = 'mobilenet'    # <-- change for backbone experiment

CLASS_NAMES = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway',
    'Industrial', 'Pasture', 'PermanentCrop', 'Residential',
    'River', 'SeaLake'
]

print('TensorFlow version:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))
print('Configuration loaded ✅')

## 2) Dataset Preparation

We load EuroSAT via **TensorFlow Datasets** (automatic download & caching).  
A synthetic fallback is provided so every cell can still run without internet access.

### 2.1 Load EuroSAT from TensorFlow Datasets

In [ ]:
def load_eurosat_tfds(img_size=IMG_SIZE):
    """Load EuroSAT RGB via tensorflow-datasets.

    Returns:
        images  : np.ndarray, shape (N, H, W, 3), float32 in [0, 1]
        labels  : np.ndarray, shape (N,), int
    """
    try:
        import tensorflow_datasets as tfds
    except ImportError:
        raise ImportError(
            'Run: pip install tensorflow-datasets  to download EuroSAT automatically.'
        )

    print('Downloading / loading EuroSAT via TensorFlow Datasets …')
    ds, info = tfds.load(
        'eurosat/rgb',
        split='train',
        as_supervised=True,
        with_info=True,
        shuffle_files=False,
    )

    def preprocess(img, lbl):
        img = tf.image.resize(img, img_size)
        img = tf.cast(img, tf.float32) / 255.0
        return img, lbl

    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    images, labels = [], []
    for img, lbl in ds.as_numpy_iterator():
        images.append(img)
        labels.append(int(lbl))

    print(f'Loaded {len(images)} images, {info.features["label"].num_classes} classes.')
    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.int32)


def load_eurosat_from_folder(data_path, img_size=IMG_SIZE):
    """Load EuroSAT from a local folder with sub-folders per class.

    Folder structure:
        data_path/
            AnnualCrop/  *.jpg
            Forest/      *.jpg
            …

    Returns:
        images, labels, class_names
    """
    from skimage.io import imread
    from skimage.transform import resize as sk_resize

    data_path = Path(data_path)
    classes   = sorted([d for d in data_path.iterdir() if d.is_dir()])
    class_names_local = [c.name for c in classes]

    images, labels = [], []
    for idx, cls_dir in enumerate(classes):
        files = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
        for fp in files:
            try:
                img = imread(str(fp))
                if img.ndim == 2:                    # grayscale → RGB
                    img = np.stack([img]*3, axis=-1)
                img = sk_resize(img, img_size, anti_aliasing=True).astype(np.float32)
                images.append(img)
                labels.append(idx)
            except Exception as exc:
                print(f'Skip {fp}: {exc}')

    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.int32), class_names_local


print('Dataset loader functions defined ✅')

In [ ]:
# ── Select data source ─────────────────────────────────────────────────────────
# Option A (recommended): load via TensorFlow Datasets (auto-download)
# Option B: load from a local folder  → set USE_TFDS=False and LOCAL_PATH
# Option C: synthetic demo            → set USE_SYNTHETIC=True

USE_TFDS      = True          # <-- set False if TFDS not installed
USE_SYNTHETIC = False          # <-- set True for offline demo (random data)
LOCAL_PATH    = 'data/EuroSAT_RGB'  # only used when USE_TFDS=False

if USE_SYNTHETIC:
    # ── Synthetic fallback (shapes/colours only, labels are random) ────────────
    print('Running in SYNTHETIC mode — results will not be meaningful.')
    N_SYNTH = 2000
    all_images = np.random.rand(N_SYNTH, *IMG_SHAPE).astype(np.float32)
    all_labels = np.random.randint(0, NUM_CLASSES, N_SYNTH).astype(np.int32)

elif USE_TFDS:
    all_images, all_labels = load_eurosat_tfds(IMG_SIZE)

else:
    all_images, all_labels, CLASS_NAMES = load_eurosat_from_folder(LOCAL_PATH, IMG_SIZE)

print(f'Dataset shape  : {all_images.shape}')
print(f'Labels shape   : {all_labels.shape}')
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')

### 2.2 Exploratory Data Analysis

In [ ]:
def plot_class_distribution(labels, class_names, title='Class Distribution'):
    """Bar chart of class frequencies."""
    counts = np.bincount(labels, minlength=len(class_names))
    fig, ax = plt.subplots(figsize=(12, 4))
    bars = ax.bar(class_names, counts, color=sns.color_palette('tab10', len(class_names)))
    ax.set_xlabel('Class'); ax.set_ylabel('Count')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(cnt), ha='center', va='bottom', fontsize=9)
    plt.tight_layout(); plt.show()


def plot_sample_images(images, labels, class_names, n_per_class=3):
    """Grid of sample images, one row per class."""
    n_classes = len(class_names)
    fig, axes = plt.subplots(n_classes, n_per_class,
                             figsize=(n_per_class * 2.5, n_classes * 2.5))
    for cls_idx, cls_name in enumerate(class_names):
        idxs = np.where(labels == cls_idx)[0][:n_per_class]
        for col, img_idx in enumerate(idxs):
            ax = axes[cls_idx, col]
            ax.imshow(images[img_idx])
            ax.axis('off')
            if col == 0:
                ax.set_title(cls_name, fontsize=9, fontweight='bold', loc='left')
    plt.suptitle('Sample Images per Class', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()


plot_class_distribution(all_labels, CLASS_NAMES)
plot_sample_images(all_images, all_labels, CLASS_NAMES)

### 2.3 Train / Validation / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# 70 % train | 15 % val | 15 % test  (stratified)
X_train, X_temp, y_train, y_temp = train_test_split(
    all_images, all_labels,
    test_size=0.30, random_state=RANDOM_STATE, stratify=all_labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print(f'Train : {X_train.shape[0]:>5} images')
print(f'Val   : {X_val.shape[0]:>5} images')
print(f'Test  : {X_test.shape[0]:>5} images')

## 3) Data Augmentation (Rubric Step 1)

Augmentation is applied **only to the training set** to improve generalisation:  
* Random rotation (±15 °)
* Horizontal & vertical flipping
* Random zoom (scaling, ±10 %)
* Width / height shifts
* Brightness jitter

Validation and test images receive **no augmentation** — only re-scaling (already done during loading).

In [ ]:
# ── Keras ImageDataGenerator augmentation pipeline ────────────────────────────
train_datagen = ImageDataGenerator(
    rotation_range=15,          # rubric: rotation
    horizontal_flip=True,       # rubric: flipping
    vertical_flip=True,
    zoom_range=0.10,            # rubric: scaling
    width_shift_range=0.10,
    height_shift_range=0.10,
    brightness_range=[0.85, 1.15],
    fill_mode='reflect',
)

val_test_datagen = ImageDataGenerator()   # no augmentation for val / test

# Build generators
train_gen = train_datagen.flow(
    X_train, tf.keras.utils.to_categorical(y_train, NUM_CLASSES),
    batch_size=BATCH_SIZE, seed=RANDOM_STATE, shuffle=True
)
val_gen = val_test_datagen.flow(
    X_val, tf.keras.utils.to_categorical(y_val, NUM_CLASSES),
    batch_size=BATCH_SIZE, shuffle=False
)
test_gen = val_test_datagen.flow(
    X_test, tf.keras.utils.to_categorical(y_test, NUM_CLASSES),
    batch_size=BATCH_SIZE, shuffle=False
)

print('Augmentation pipeline ready ✅')
print('Train steps per epoch:', len(train_gen))

In [ ]:
def visualise_augmentation(images, labels, class_names, datagen, n_aug=6):
    """Show one original image alongside several augmented versions."""
    idx = np.random.randint(len(images))
    orig = images[idx]
    cls  = class_names[labels[idx]]

    aug_iter = datagen.flow(
        orig[np.newaxis], batch_size=1, seed=RANDOM_STATE
    )

    fig, axes = plt.subplots(1, n_aug + 1, figsize=(2.5 * (n_aug + 1), 3))
    axes[0].imshow(orig); axes[0].set_title('Original\n' + cls, fontsize=9); axes[0].axis('off')
    for i in range(n_aug):
        aug = next(aug_iter)[0]
        axes[i + 1].imshow(aug.clip(0, 1))
        axes[i + 1].set_title(f'Aug {i+1}', fontsize=9)
        axes[i + 1].axis('off')
    plt.suptitle('Data Augmentation Examples', fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()


visualise_augmentation(X_train, y_train, CLASS_NAMES, train_datagen)

## 4) CNN Model Design (Rubric Step 2)

We define **two model factories**:

| Model | Description |
|---|---|
| `build_custom_cnn` | 4-block custom CNN built from scratch |
| `build_transfer_model` | MobileNetV2 (ImageNet weights) with a new classification head |

Switch between them using the `MODEL_TYPE` flag.

In [ ]:
def build_custom_cnn(input_shape=IMG_SHAPE, num_classes=NUM_CLASSES, dropout=0.4):
    """Custom 4-block CNN.

    Architecture:
        Conv-BN-ReLU → Conv-BN-ReLU → MaxPool → Dropout  (x4 blocks)
        GlobalAveragePooling → Dense(256) → Dropout → Dense(num_classes)

    Design choices:
        - Batch Normalisation for stable training
        - Increasing filter counts (32→64→128→256) to capture richer features
        - Global Average Pooling instead of Flatten to reduce parameters
        - L2 regularisation on dense layers to reduce overfitting
    """
    inp = keras.Input(shape=input_shape, name='input')

    x = inp
    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(filters, (3, 3), padding='same', activation='relu',
                          kernel_regularizer=regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(filters, (3, 3), padding='same', activation='relu',
                          kernel_regularizer=regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x)
        x = layers.Dropout(dropout * 0.5)(x)   # spatial dropout before pooling

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = keras.Model(inp, out, name='CustomCNN')
    return model


# ── Supported transfer-learning backbones ─────────────────────────────────────
# Backbone   | Min input   | Notes
# -----------|-------------|------------------------------------------------------
# mobilenet  | 32×32       | Lightweight; fast training
# resnet     | 32×32       | ResNet50V2; deep residual connections
# vgg        | 48×48       | VGG16; classic architecture
# efficientnet| 32×32      | EfficientNetB0; compound scaling

_BACKBONE_REGISTRY = {
    'mobilenet': {
        'cls':        MobileNetV2,
        'preprocess': tf.keras.applications.mobilenet_v2.preprocess_input,
        'name':       'MobileNetV2',
    },
    'resnet': {
        'cls':        ResNet50V2,
        'preprocess': tf.keras.applications.resnet_v2.preprocess_input,
        'name':       'ResNet50V2',
    },
    'vgg': {
        'cls':        VGG16,
        'preprocess': tf.keras.applications.vgg16.preprocess_input,
        'name':       'VGG16',
    },
    'efficientnet': {
        'cls':        EfficientNetB0,
        'preprocess': tf.keras.applications.efficientnet.preprocess_input,
        'name':       'EfficientNetB0',
    },
}


def build_transfer_model(input_shape=IMG_SHAPE, num_classes=NUM_CLASSES,
                          dropout=0.4, fine_tune_from=None,
                          backbone=BACKBONE):
    """Transfer-learning model with a selectable ImageNet-pretrained backbone.

    Supported backbones (set via BACKBONE global or 'backbone' arg):
        'mobilenet'    → MobileNetV2   (lightweight, fast)
        'resnet'       → ResNet50V2    (deep residual connections)
        'vgg'          → VGG16         (classic VGG architecture)
        'efficientnet' → EfficientNetB0 (compound-scaled efficiency)

    Args:
        fine_tune_from: if int, unfreeze layers from that index onwards
                        (None = feature extraction only, backbone frozen).
        backbone: key into _BACKBONE_REGISTRY (default: BACKBONE global).

    Architecture:
        <Backbone> (frozen / partially unfrozen)
        → GlobalAveragePooling → BN → Dense(256) → Dropout → Dense(num_classes)
    """
    cfg = _BACKBONE_REGISTRY.get(backbone.lower())
    if cfg is None:
        raise ValueError(
            f"Unknown backbone '{backbone}'. "
            f"Choose: {list(_BACKBONE_REGISTRY.keys())}"
        )

    base = cfg['cls'](
        input_shape=input_shape,
        include_top=False,
        weights='imagenet',
    )
    base.trainable = False

    if fine_tune_from is not None:
        base.trainable = True
        for layer in base.layers[:fine_tune_from]:
            layer.trainable = False

    inp = keras.Input(shape=input_shape, name='input')
    x   = cfg['preprocess'](inp * 255.0)   # scale [0,1] → [0,255] expected by most preprocessors
    x   = base(x, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dense(256, activation='relu',
                        kernel_regularizer=regularizers.l2(1e-4))(x)
    x   = layers.Dropout(dropout)(x)
    out = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = keras.Model(inp, out, name=f'{cfg["name"]}_Transfer')
    return model


print('Model factories defined ✅')
print(f'Available backbones: {list(_BACKBONE_REGISTRY.keys())}')

In [ ]:
# ── Select model for this run ──────────────────────────────────────────────────
# MODEL_TYPE options : 'custom'  |  'transfer'  |  'transfer_finetune'
# BACKBONE options   : 'mobilenet' | 'resnet' | 'vgg' | 'efficientnet'
#   (BACKBONE is set in §1 — change it there to run the backbone experiment)
MODEL_TYPE = 'custom'        # <-- change for architecture experiment

if MODEL_TYPE == 'custom':
    model = build_custom_cnn()
elif MODEL_TYPE == 'transfer':
    model = build_transfer_model(backbone=BACKBONE)
elif MODEL_TYPE == 'transfer_finetune':
    # Unfreeze the last ~30 % of backbone layers for fine-tuning
    backbone_obj = _BACKBONE_REGISTRY[BACKBONE.lower()]['cls'](
        input_shape=IMG_SHAPE, include_top=False, weights='imagenet'
    )
    fine_tune_at = int(len(backbone_obj.layers) * 0.70)
    model = build_transfer_model(backbone=BACKBONE, fine_tune_from=fine_tune_at)
else:
    raise ValueError(f'Unknown MODEL_TYPE: {MODEL_TYPE}')

model.summary()

## 5) Model Training (Rubric Step 3)

### Configurable Optimizer

The optimizer is selected by `OPTIMIZER_NAME` (set in §1).  
Change it to `'sgd'`, `'rmsprop'`, or `'adamw'` and re-run this section for the optimizer tuning experiment.

In [ ]:
def get_optimizer(name: str, lr: float = 1e-3):
    """Factory for Keras optimizers (configurable for experiments)."""
    name = name.lower()
    if name == 'adam':
        return keras.optimizers.Adam(learning_rate=lr)
    elif name == 'sgd':
        return keras.optimizers.SGD(learning_rate=lr, momentum=0.9, nesterov=True)
    elif name == 'rmsprop':
        return keras.optimizers.RMSprop(learning_rate=lr, rho=0.9)
    elif name == 'adamw':
        return keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4)
    else:
        raise ValueError(f'Unknown optimizer: {name}')


optimizer = get_optimizer(OPTIMIZER_NAME, LEARNING_RATE)
print(f'Optimizer : {OPTIMIZER_NAME} (lr={LEARNING_RATE})')

model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print('Model compiled ✅')

In [ ]:
# ── Callbacks ─────────────────────────────────────────────────────────────────
os.makedirs('models', exist_ok=True)
os.makedirs('logs',   exist_ok=True)

checkpoint_path = f'models/best_{MODEL_TYPE}_{OPTIMIZER_NAME}.keras'
log_path        = f'logs/training_{MODEL_TYPE}_{OPTIMIZER_NAME}.csv'

callbacks = [
    # Stop early if val_loss does not improve for 7 epochs
    EarlyStopping(
        monitor='val_loss', patience=7, restore_best_weights=True, verbose=1
    ),
    # Reduce LR on plateau
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
    # Save best model
    ModelCheckpoint(
        checkpoint_path, monitor='val_accuracy',
        save_best_only=True, verbose=0
    ),
    # Log metrics to CSV for post-analysis
    CSVLogger(log_path, append=False),
]

print(f'Checkpoint : {checkpoint_path}')
print(f'CSV log    : {log_path}')

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
t0 = time.time()

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

elapsed = time.time() - t0
print(f'\nTraining complete in {elapsed/60:.1f} min  ✅')
print(f'Best val accuracy  : {max(history.history["val_accuracy"]):.4f}')

## 6) Performance Evaluation (Rubric Step 4)

Metrics computed on the **held-out test set** (never seen during training):
- Overall accuracy
- Per-class precision, recall, F1
- Macro-averaged precision, recall, F1
- Confusion matrix
- Learning curves (train vs. val accuracy & loss)

### 6.1 Learning Curves

In [ ]:
def plot_learning_curves(history, title='Learning Curves'):
    """Plot training / validation accuracy and loss side by side."""
    epochs_ran = range(1, len(history.history['loss']) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    ax1.plot(epochs_ran, history.history['accuracy'],     label='Train')
    ax1.plot(epochs_ran, history.history['val_accuracy'], label='Validation')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(True, alpha=0.4)

    # Loss
    ax2.plot(epochs_ran, history.history['loss'],     label='Train')
    ax2.plot(epochs_ran, history.history['val_loss'], label='Validation')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.set_title('Loss'); ax2.legend(); ax2.grid(True, alpha=0.4)

    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()


plot_learning_curves(history, title=f'Learning Curves — {MODEL_TYPE} / {OPTIMIZER_NAME}')

### 6.2 Test-Set Predictions

In [ ]:
# Generate predictions on the full test set
y_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
y_pred = np.argmax(y_prob, axis=1)     # predicted class indices
y_true = y_test                         # ground-truth class indices

acc   = accuracy_score(y_true, y_pred)
prec  = precision_score(y_true, y_pred, average='macro', zero_division=0)
rec   = recall_score(y_true, y_pred,    average='macro', zero_division=0)
f1    = f1_score(y_true, y_pred,        average='macro', zero_division=0)

print('=' * 50)
print(f'  Test Accuracy  : {acc:.4f}')
print(f'  Macro Precision: {prec:.4f}')
print(f'  Macro Recall   : {rec:.4f}')
print(f'  Macro F1-Score : {f1:.4f}')
print('=' * 50)

### 6.3 Per-Class Classification Report

In [ ]:
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

### 6.4 Confusion Matrix

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, title='Confusion Matrix', normalize=True):
    """Plot a (optionally normalized) confusion matrix as a heatmap."""
    cm = confusion_matrix(y_true, y_pred)
    if normalize:
        cm_disp = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        fmt = '.2f'
    else:
        cm_disp = cm
        fmt = 'd'

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        cm_disp, annot=True, fmt=fmt, cmap='Blues',
        xticklabels=class_names, yticklabels=class_names, ax=ax
    )
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('True', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)
    plt.tight_layout(); plt.show()


plot_confusion_matrix(
    y_true, y_pred, CLASS_NAMES,
    title=f'Confusion Matrix (normalized) — {MODEL_TYPE} / {OPTIMIZER_NAME}'
)

### 6.5 Per-Class Metrics Bar Chart

In [ ]:
def plot_per_class_metrics(y_true, y_pred, class_names):
    """Grouped bar chart of per-class precision, recall, and F1."""
    prec_c  = precision_score(y_true, y_pred, average=None, zero_division=0)
    rec_c   = recall_score(y_true, y_pred,    average=None, zero_division=0)
    f1_c    = f1_score(y_true, y_pred,        average=None, zero_division=0)

    x   = np.arange(len(class_names))
    w   = 0.25

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(x - w, prec_c, w, label='Precision', color='steelblue')
    ax.bar(x,     rec_c,  w, label='Recall',    color='darkorange')
    ax.bar(x + w, f1_c,   w, label='F1',        color='seagreen')

    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Per-Class Metrics (Test Set)', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.4)
    plt.tight_layout(); plt.show()


plot_per_class_metrics(y_true, y_pred, CLASS_NAMES)

### 6.6 Prediction Visualisation — Correct vs. Incorrect Samples

In [ ]:
def plot_predictions(images, y_true, y_pred, y_prob, class_names,
                     n=8, correct_only=None, title='Predictions'):
    """Show images with their true and predicted labels, colour-coded.

    Args:
        correct_only: True → show only correct, False → show only errors, None → mixed
    """
    mask = np.ones(len(images), dtype=bool)
    if correct_only is True:
        mask = y_true == y_pred
    elif correct_only is False:
        mask = y_true != y_pred

    idxs = np.where(mask)[0]
    if len(idxs) == 0:
        print('No matching samples found.'); return

    rng  = np.random.default_rng(RANDOM_STATE)
    idxs = rng.choice(idxs, size=min(n, len(idxs)), replace=False)

    cols = min(n, 4)
    rows = (len(idxs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = np.array(axes).reshape(-1)

    for ax, i in zip(axes, idxs):
        ax.imshow(images[i])
        conf = y_prob[i, y_pred[i]]
        colour = 'green' if y_true[i] == y_pred[i] else 'red'
        ax.set_title(
            f'True: {class_names[y_true[i]]}\n'
            f'Pred: {class_names[y_pred[i]]} ({conf:.2f})',
            fontsize=8, color=colour
        )
        ax.axis('off')

    for ax in axes[len(idxs):]:
        ax.axis('off')

    plt.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()


plot_predictions(X_test, y_true, y_pred, y_prob, CLASS_NAMES,
                 n=8, correct_only=True,  title='Correctly Classified Samples')
plot_predictions(X_test, y_true, y_pred, y_prob, CLASS_NAMES,
                 n=8, correct_only=False, title='Misclassified Samples')

## 7) Save Results

In [ ]:
os.makedirs('reports', exist_ok=True)

# ── Scalar metrics → JSON ──────────────────────────────────────────────────────
results = {
    'model_type'   : MODEL_TYPE,
    'optimizer'    : OPTIMIZER_NAME,
    'learning_rate': LEARNING_RATE,
    'epochs_run'   : len(history.history['loss']),
    'test_accuracy': float(acc),
    'test_precision_macro': float(prec),
    'test_recall_macro'   : float(rec),
    'test_f1_macro'       : float(f1),
    'best_val_accuracy'   : float(max(history.history['val_accuracy'])),
    'confusion_matrix'    : confusion_matrix(y_true, y_pred).tolist(),
}

json_path = f'reports/results_{MODEL_TYPE}_{OPTIMIZER_NAME}.json'
with open(json_path, 'w') as fh:
    json.dump(results, fh, indent=4)

print(f'Results saved → {json_path}')

# ── Per-epoch CSV (already written by CSVLogger) ───────────────────────────────
print(f'Training log  → {log_path}')

## 8) Experimental Comparison

After running the notebook multiple times with different settings, aggregate results here.

In [ ]:
def load_experiment_results(report_dir='reports'):
    """Load all JSON result files and display a comparison table."""
    rows = []
    for fp in Path(report_dir).glob('results_*.json'):
        with open(fp) as fh:
            r = json.load(fh)
        rows.append({
            'Experiment'       : fp.stem.replace('results_', ''),
            'Model'            : r.get('model_type', ''),
            'Optimizer'        : r.get('optimizer', ''),
            'LR'               : r.get('learning_rate', ''),
            'Epochs'           : r.get('epochs_run', ''),
            'Test Accuracy'    : round(r.get('test_accuracy', 0), 4),
            'Macro F1'         : round(r.get('test_f1_macro', 0), 4),
            'Macro Precision'  : round(r.get('test_precision_macro', 0), 4),
            'Macro Recall'     : round(r.get('test_recall_macro', 0), 4),
            'Best Val Accuracy': round(r.get('best_val_accuracy', 0), 4),
        })
    if not rows:
        print('No result files found yet. Run experiments first.')
        return pd.DataFrame()
    df = pd.DataFrame(rows).sort_values('Test Accuracy', ascending=False)
    return df


df_results = load_experiment_results()
if not df_results.empty:
    display(df_results)

In [ ]:
def plot_experiment_comparison(df_results):
    """Bar chart comparing test accuracy across experiments."""
    if df_results.empty:
        return

    fig, ax = plt.subplots(figsize=(max(8, len(df_results) * 1.5), 5))
    colours = sns.color_palette('tab10', len(df_results))
    bars = ax.bar(df_results['Experiment'], df_results['Test Accuracy'], color=colours)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Test Accuracy')
    ax.set_title('Experiment Comparison — Test Accuracy', fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', alpha=0.4)
    for bar, (_, row) in zip(bars, df_results.iterrows()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{row['Test Accuracy']:.3f}", ha='center', va='bottom', fontsize=9)
    plt.tight_layout(); plt.show()


plot_experiment_comparison(df_results)

## 9) Inference Utility — Predict on a Single New Image

In [ ]:
def predict_single_image(image_path_or_array, model, class_names,
                          img_size=IMG_SIZE):
    """Load (or accept) one image and print the top-3 predicted classes.

    Args:
        image_path_or_array: str path OR np.ndarray (H, W, 3) float32 in [0, 1]
    """
    from skimage.io import imread
    from skimage.transform import resize as sk_resize

    if isinstance(image_path_or_array, (str, Path)):
        img = imread(str(image_path_or_array)).astype(np.float32) / 255.0
        img = sk_resize(img, img_size, anti_aliasing=True)
    else:
        img = image_path_or_array

    if img.ndim == 2:                           # grayscale → RGB
        img = np.stack([img]*3, axis=-1)

    prob  = model.predict(img[np.newaxis], verbose=0)[0]   # (num_classes,)
    top3  = np.argsort(prob)[::-1][:3]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img.clip(0, 1))
    axes[0].set_title('Input Image', fontsize=11)
    axes[0].axis('off')

    colours_bar = ['#2196F3' if i == top3[0] else '#90CAF9' for i in range(len(class_names))]
    axes[1].barh(class_names, prob, color=colours_bar)
    axes[1].set_xlim(0, 1)
    axes[1].set_xlabel('Probability')
    axes[1].set_title(f'Prediction: {class_names[top3[0]]} ({prob[top3[0]]:.2f})', fontsize=11)
    axes[1].invert_yaxis()
    axes[1].grid(axis='x', alpha=0.4)

    plt.tight_layout(); plt.show()

    print('Top-3 predictions:')
    for rank, idx in enumerate(top3, 1):
        print(f'  {rank}. {class_names[idx]:30s} {prob[idx]:.4f}')

    return class_names[top3[0]], prob


# ── Demo: predict on a random test image ──────────────────────────────────────
demo_idx  = np.random.randint(len(X_test))
demo_img  = X_test[demo_idx]
pred_cls, pred_prob = predict_single_image(demo_img, model, CLASS_NAMES)
print(f'\nGround truth : {CLASS_NAMES[y_test[demo_idx]]}')
print(f'Predicted    : {pred_cls}')

## 10) Summary & Experimental Plan (Report Guide)

### Summary
This notebook demonstrates a complete CNN-based image classification pipeline on the **EuroSAT** satellite imagery benchmark (10 land-use classes, 27 000 RGB images).  
Key components:

| Component | Detail |
|---|---|
| Dataset | EuroSAT RGB — Sentinel-2 patches, 64×64 px, 10 classes |
| Augmentation | Rotation ±15°, H/V flip, zoom ±10 %, shift ±10 %, brightness jitter |
| Custom CNN | 4 conv-blocks (32→64→128→256), BN, GAP, Dense(256), Dropout |
| Transfer Learning | MobileNetV2 / ResNet50V2 / VGG16 / EfficientNetB0 (ImageNet) + BN + Dense(256) + Dropout |
| Optimizers | Adam, SGD+momentum, RMSprop, AdamW (configurable) |
| Metrics | Accuracy, Precision, Recall, F1 (macro), Confusion Matrix, Learning Curves |

---

### Experimental Plan for the Report

Run the following 5 experiments and fill in the table:

| # | `MODEL_TYPE` | `OPTIMIZER_NAME` | Augmentation | Notes |
|---|---|---|---|---|
| E1 | `custom` | `adam` | OFF (set `train_datagen = ImageDataGenerator()`) | Baseline |
| E2 | `custom` | `adam` | ON | Augmentation effect |
| E3 | `transfer` | `adam` | ON | Transfer learning |
| E4 | `transfer` | `sgd` | ON | Optimizer comparison |
| E5 | `transfer_finetune` | `adam` | ON | Fine-tuning effect |

**Research questions:**
1. How much does data augmentation improve test accuracy on satellite images?
2. Does transfer learning (MobileNetV2) outperform a custom CNN, and by how much?
3. Which optimizer converges faster and/or achieves higher accuracy?
4. Does fine-tuning the backbone further improve results over feature extraction only?

Use `§8` to compare all saved JSON results automatically.